In [1]:
%%time

import geopandas as gpd
import pandas as pd

gdf = gpd.read_file(r'../../datafiles/processed/260514_India_DistrictGap_v1.gpkg')
print(gdf.shape)
gdf.head()

(735, 10)


,state23,st_code,GeoID,GeoName,population,perceived,imd_flood_count,assessed,gap,geometry
0,ANDAMAN & NICOBAR,35,35603.0,"Andaman & Nicobar, Nicobar",26697.0,49.083213,0.0,0.000000,-49.083213,"MULTIPOLYGON (((93.95032 6.93412, 93.95018 6.9..."
1,ANDAMAN & NICOBAR,35,35632.0,"Andaman & Nicobar, North & Middle Andaman",71857.0,41.220610,1.0,1.587302,-39.633308,"MULTIPOLYGON (((92.79855 12.07663, 92.79868 12..."
2,ANDAMAN & NICOBAR,35,35602.0,"Andaman & Nicobar, South Andaman",169248.0,75.111021,1.0,1.587302,-73.523719,"MULTIPOLYGON (((92.51836 10.90084, 92.51906 10..."
3,ANDHRA PRADESH,28,28502.0,"Andhra Pradesh, Anantapur",2757734.0,41.240960,30.0,47.619048,6.378088,"MULTIPOLYGON (((77.13122 15.42632, 77.13199 15..."
4,ANDHRA PRADESH,28,28503.0,"Andhra Pradesh, Chittoor",2864035.0,45.562078,35.0,55.555556,9.993478,"MULTIPOLYGON (((78.65631 13.90609, 78.65625 13..."


In [16]:
%%time

import pandas as pd

df = pd.read_csv(r'../../datafiles/260514_all_covariates_merged_surabhi.csv')

# add GeoID to match with the shapefile
df['GeoID'] = df.apply(lambda x: f'{x.st_code:02d}{x.di_code:03d}', axis=1).astype(float)
df.drop(columns=['di_code', 'dist23', 'st_code', 'state23', 'pred_per_diff'], inplace=True)

print(df.shape)
df.head()

(729, 44)
CPU times: total: 31.2 ms
Wall time: 25.9 ms


,FlowAcc_sum,GFD_sum,mean_monthly_avg_rd_MD,mean_monthly_avg_rd_MN,precip_sum_mm,precip_mean_mm,mean_tmax,highersecondaryabove_district,vulnerability_district,anger,...,anger_intensity,fear_intensity,sadness_intensity,joy_intensity,disgust_intensity,surprise_intensity,anticipation_intensity,trust_intensity,flood_news_count,GeoID
0,1.578700e+06,1051.780392,0.592195,0.560304,213.092819,53.273205,13.874764,-0.360831,1.872751,3.0,...,0.013043,0.013043,0.008696,0.008696,0.004348,0.004348,0.026087,0.017391,1.0,1008.0
1,8.022186e+05,3075.603922,1.522485,1.467856,250.073044,83.357681,15.272575,-0.320944,-0.816196,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1002.0
2,1.004582e+07,247.125490,0.963362,0.962803,268.853333,53.770666,19.800248,-0.228072,-0.816196,3.0,...,0.012931,0.025862,0.017241,0.004310,0.000000,0.004310,0.012931,0.012931,1.0,1010.0
3,1.719527e+06,20.823529,1.095254,1.120407,217.372925,54.343231,23.025746,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1012.0
4,3.559943e+06,7526.807843,1.283567,1.282422,361.107269,72.221454,23.646945,-0.153114,-0.816196,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1007.0


# Merge the two dataframes

In [21]:
out_gdf = gdf.merge(df, on='GeoID', how='left')
out_gdf.head()

,state23,st_code,GeoID,GeoName,population,perceived,imd_flood_count,assessed,gap,geometry,...,trust_share,anger_intensity,fear_intensity,sadness_intensity,joy_intensity,disgust_intensity,surprise_intensity,anticipation_intensity,trust_intensity,flood_news_count
0,ANDAMAN & NICOBAR,35,35603.0,"Andaman & Nicobar, Nicobar",26697.0,49.083213,0.0,0.000000,-49.083213,"MULTIPOLYGON (((93.95032 6.93412, 93.95018 6.9...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ANDAMAN & NICOBAR,35,35632.0,"Andaman & Nicobar, North & Middle Andaman",71857.0,41.220610,1.0,1.587302,-39.633308,"MULTIPOLYGON (((92.79855 12.07663, 92.79868 12...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANDAMAN & NICOBAR,35,35602.0,"Andaman & Nicobar, South Andaman",169248.0,75.111021,1.0,1.587302,-73.523719,"MULTIPOLYGON (((92.51836 10.90084, 92.51906 10...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANDHRA PRADESH,28,28502.0,"Andhra Pradesh, Anantapur",2757734.0,41.240960,30.0,47.619048,6.378088,"MULTIPOLYGON (((77.13122 15.42632, 77.13199 15...",...,0.185185,0.008475,0.038136,0.000000,0.000000,0.000000,0.029661,0.016949,0.021186,1.0
4,ANDHRA PRADESH,28,28503.0,"Andhra Pradesh, Chittoor",2864035.0,45.562078,35.0,55.555556,9.993478,"MULTIPOLYGON (((78.65631 13.90609, 78.65625 13...",...,0.186147,0.009147,0.015318,0.009147,0.004792,0.002976,0.001597,0.010963,0.012342,2.0


In [22]:
out_gdf.to_file(r'../../datafiles/processed/260515_India_DistrictGap_WithCovariates_v2.gpkg')